# Notebook 14 — StratLake Campaign Evidence Review Pack and Governance Audit

This is a raw pre-import research/development draft for `christophermoverton/fintech-stratlake-notebook-workflows`.

**Theme:** From native campaign execution artifacts to conservative evidence review and governance audit.

Notebook 14 follows Notebook 13. Notebook 13 performs guarded native StratLake campaign execution when explicitly enabled. Notebook 14 does **not** rerun or reinterpret campaign execution by default. Instead, it restores or inspects campaign artifacts, optionally builds native derived evidence review packs, optionally runs native promotion governance reporting, inventories the resulting artifacts, records caveats, and prepares a human-review handoff.

**Target path:** `notebooks/14_stratlake_campaign_evidence_review_pack_and_governance_audit.ipynb`

**Milestone:** M17 — Notebook 14 Campaign Evidence Review Pack and Governance Audit Import

**Conservative default stance:** `notebook_14_staged_cleaned_source_safe_evidence_governance_audit`


## Source-safe and native-command-first posture

`stratlake-trade-engine` remains the source of truth for campaign artifacts, evidence review packs, promotion governance reports, catalog indexing, lineage export, archive restore/checkpoint behavior, and promotion semantics.

This notebook may discover native command surfaces, inspect command availability/help text, restore a prior Notebook 13 session only when explicitly enabled, discover campaign artifacts, build/validate derived evidence review packs only when explicitly enabled, run promotion governance reporting only when explicitly enabled, index/query/export catalog and lineage surfaces only when explicitly enabled, and write runtime handoff summaries outside Git.

This notebook must not rerun campaign execution by default, reimplement evidence review or governance semantics, treat derived review packs as canonical artifacts, fabricate missing governance rows or split metrics, write runtime artifacts into Git-tracked source paths, or claim strategy approval, promotion readiness, governance readiness, alpha validation, statistical significance, production readiness, artifact completeness, source/runtime equivalence, or split-metric completeness unless native evidence explicitly supports those claims.


## Relationship to Notebook 13

Notebook 13 finalized the native campaign execution and artifact generation workflow. Notebook 14 addresses the review/audit gaps carried forward after Notebook 13.

| Notebook 13 output or caveat | Notebook 14 response |
|---|---|
| Native campaign artifacts may exist outside Git | Discover and inventory candidate artifacts |
| Artifact discovery alone is not proof of current-session execution | Preserve caveats and execution-source distinctions |
| Notebook 14 runtime summaries may contain the word governance | Exclude notebook-runtime paths from campaign/governance discovery |
| Governance rows may be missing or empty | Run/read native governance report only when explicitly enabled |
| Split metrics may be missing or incomplete | Record split-metric caveats without fabricating readiness |
| Evidence review packs are optional derived artifacts | Build/validate native review packs only when explicitly enabled |
| Catalog/lineage surfaces may help review | Query/export native catalog lineage only when explicitly enabled |
| Notebook-generated summaries are non-canonical | Write runtime-only handoff summaries with non-claim language |


## 1. Install notebook dependencies and app packages


In [ ]:
!pip install -q "pandas-market-calendars>=5.0"
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine


## 2. Optional runtime toggles

The committed notebook is preview-only. Uncomment exactly one profile override in an executed runtime copy, then restart/run all.


In [ ]:
import os

# Profile override examples for runtime smoke runs.
# Keep the committed fallback in the profile selector as evidence_governance_preview.

# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_governance_preflight"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "campaign_artifact_discovery"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "archive_restore_discovery"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_review_pack_build"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "governance_report_run"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "catalog_lineage_review"
# os.environ["NOTEBOOK14_TEST_PROFILE"] = "evidence_governance_full_review"

# Optional native preflight initialization:
# os.environ["NOTEBOOK14_ALLOW_STRATLAKE_INIT"] = "true"
# os.environ["RUN_STRATLAKE_INIT"] = "true"

# Optional restore from a prior Notebook 13 session archive:
# os.environ["NOTEBOOK14_ALLOW_ARCHIVE_RESTORE"] = "true"
# os.environ["RUN_ARCHIVE_RESTORE"] = "true"
# os.environ["NOTEBOOK14_RESTORE_ARCHIVE_ID"] = "notebook-session-001"
# os.environ["NOTEBOOK14_DRIVE_ARCHIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab/session_archives"

# Optional native derived evidence review pack:
# os.environ["NOTEBOOK14_ALLOW_EVIDENCE_REVIEW"] = "true"
# os.environ["RUN_EVIDENCE_REVIEW_PACK_BUILD"] = "true"
# os.environ["RUN_EVIDENCE_REVIEW_PACK_VALIDATE"] = "true"
# os.environ["NOTEBOOK14_SELECTED_RUN_ID"] = "strategy_001"

# Optional native promotion governance report:
# os.environ["NOTEBOOK14_ALLOW_GOVERNANCE_REPORT"] = "true"
# os.environ["RUN_PROMOTION_GOVERNANCE_REPORT"] = "true"

# Optional catalog/lineage export:
# os.environ["NOTEBOOK14_ALLOW_CATALOG_LINEAGE"] = "true"
# os.environ["RUN_CATALOG_LINEAGE_EXPORT"] = "true"

# Optional archive checkpoint after review:
# os.environ["NOTEBOOK14_ALLOW_ARCHIVE_CHECKPOINT"] = "true"
# os.environ["RUN_ARCHIVE_CHECKPOINT"] = "true"

# Optional structured install warning note:
# os.environ["NOTEBOOK14_INSTALL_RESOLVER_WARNING"] = "true"


## 3. Imports, Colab detection, and display helpers


In [ ]:
import csv
import importlib
import importlib.metadata as importlib_metadata
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    userdata = None
    IN_COLAB = False

INSTALL_CAVEATS: list[str] = []
if os.environ.get("NOTEBOOK14_INSTALL_RESOLVER_WARNING", "false").lower() == "true":
    INSTALL_CAVEATS.append("Install resolver warning was observed during package installation; monitor dependency compatibility before relying on affected optional surfaces.")

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def display_df(df: pd.DataFrame, max_rows: int = 20) -> None:
    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))

def display_markdown(text: str) -> None:
    if display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def command_available(command: str) -> bool:
    return shutil.which(command) is not None

def tail_text(text: str | None, max_chars: int = 4000) -> str:
    if not text:
        return ""
    return text[-max_chars:]

def safe_json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, datetime):
        return value.isoformat()
    return str(value)

def write_json(path: Path, data: Any) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(data, indent=2, default=safe_json_default), encoding="utf-8")
    return path

def write_dataframe_csv(path: Path, df: pd.DataFrame) -> Path:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)
    return path


## 4. Runtime controls and execution profiles

The committed default profile is intentionally safe: `evidence_governance_preview`.


In [ ]:
NOTEBOOK14_TEST_PROFILE = os.environ.get("NOTEBOOK14_TEST_PROFILE", "evidence_governance_preview").strip() or "evidence_governance_preview"

PROFILE_MATRIX: dict[str, dict[str, bool]] = {
    "evidence_governance_preview": {"RUN_STRATLAKE_INIT": False, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "evidence_governance_preflight": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "campaign_artifact_discovery": {"RUN_STRATLAKE_INIT": False, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "archive_restore_discovery": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": True, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "evidence_review_pack_build": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": True, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": True, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "governance_report_run": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": True, "RUN_CATALOG_LINEAGE_EXPORT": False, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "catalog_lineage_review": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": False, "RUN_EVIDENCE_REVIEW_PACK_BUILD": False, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": False, "RUN_PROMOTION_GOVERNANCE_REPORT": False, "RUN_CATALOG_LINEAGE_EXPORT": True, "RUN_ARCHIVE_CHECKPOINT": False, "WRITE_RUNTIME_SUMMARY": True},
    "evidence_governance_full_review": {"RUN_STRATLAKE_INIT": True, "DISCOVER_NATIVE_COMMANDS": True, "DISCOVER_CAMPAIGN_ARTIFACTS": True, "RUN_ARCHIVE_RESTORE": True, "RUN_EVIDENCE_REVIEW_PACK_BUILD": True, "RUN_EVIDENCE_REVIEW_PACK_VALIDATE": True, "RUN_PROMOTION_GOVERNANCE_REPORT": True, "RUN_CATALOG_LINEAGE_EXPORT": True, "RUN_ARCHIVE_CHECKPOINT": True, "WRITE_RUNTIME_SUMMARY": True},
}

if NOTEBOOK14_TEST_PROFILE not in PROFILE_MATRIX:
    raise ValueError(f"Unknown NOTEBOOK14_TEST_PROFILE={NOTEBOOK14_TEST_PROFILE!r}")

PROFILE = PROFILE_MATRIX[NOTEBOOK14_TEST_PROFILE]
ALLOW_STRATLAKE_INIT = os.environ.get("NOTEBOOK14_ALLOW_STRATLAKE_INIT", "false").lower() == "true"
ALLOW_ARCHIVE_RESTORE = os.environ.get("NOTEBOOK14_ALLOW_ARCHIVE_RESTORE", "false").lower() == "true"
ALLOW_EVIDENCE_REVIEW = os.environ.get("NOTEBOOK14_ALLOW_EVIDENCE_REVIEW", "false").lower() == "true"
ALLOW_GOVERNANCE_REPORT = os.environ.get("NOTEBOOK14_ALLOW_GOVERNANCE_REPORT", "false").lower() == "true"
ALLOW_CATALOG_LINEAGE = os.environ.get("NOTEBOOK14_ALLOW_CATALOG_LINEAGE", "false").lower() == "true"
ALLOW_ARCHIVE_CHECKPOINT = os.environ.get("NOTEBOOK14_ALLOW_ARCHIVE_CHECKPOINT", "false").lower() == "true"
RUN_STRATLAKE_INIT = PROFILE["RUN_STRATLAKE_INIT"] and ALLOW_STRATLAKE_INIT and os.environ.get("RUN_STRATLAKE_INIT", "false").lower() == "true"
RUN_ARCHIVE_RESTORE = PROFILE["RUN_ARCHIVE_RESTORE"] and ALLOW_ARCHIVE_RESTORE and os.environ.get("RUN_ARCHIVE_RESTORE", "false").lower() == "true"
RUN_EVIDENCE_REVIEW_PACK_BUILD = PROFILE["RUN_EVIDENCE_REVIEW_PACK_BUILD"] and ALLOW_EVIDENCE_REVIEW and os.environ.get("RUN_EVIDENCE_REVIEW_PACK_BUILD", "false").lower() == "true"
RUN_EVIDENCE_REVIEW_PACK_VALIDATE = PROFILE["RUN_EVIDENCE_REVIEW_PACK_VALIDATE"] and ALLOW_EVIDENCE_REVIEW and os.environ.get("RUN_EVIDENCE_REVIEW_PACK_VALIDATE", "false").lower() == "true"
RUN_PROMOTION_GOVERNANCE_REPORT = PROFILE["RUN_PROMOTION_GOVERNANCE_REPORT"] and ALLOW_GOVERNANCE_REPORT and os.environ.get("RUN_PROMOTION_GOVERNANCE_REPORT", "false").lower() == "true"
RUN_CATALOG_LINEAGE_EXPORT = PROFILE["RUN_CATALOG_LINEAGE_EXPORT"] and ALLOW_CATALOG_LINEAGE and os.environ.get("RUN_CATALOG_LINEAGE_EXPORT", "false").lower() == "true"
RUN_ARCHIVE_CHECKPOINT = PROFILE["RUN_ARCHIVE_CHECKPOINT"] and ALLOW_ARCHIVE_CHECKPOINT and os.environ.get("RUN_ARCHIVE_CHECKPOINT", "false").lower() == "true"
runtime_controls = {"profile": NOTEBOOK14_TEST_PROFILE, "source_safe_default": NOTEBOOK14_TEST_PROFILE == "evidence_governance_preview", "profile_requests_stratlake_init": PROFILE["RUN_STRATLAKE_INIT"], "run_stratlake_init": RUN_STRATLAKE_INIT, "run_archive_restore": RUN_ARCHIVE_RESTORE, "run_evidence_review_pack_build": RUN_EVIDENCE_REVIEW_PACK_BUILD, "run_evidence_review_pack_validate": RUN_EVIDENCE_REVIEW_PACK_VALIDATE, "run_promotion_governance_report": RUN_PROMOTION_GOVERNANCE_REPORT, "run_catalog_lineage_export": RUN_CATALOG_LINEAGE_EXPORT, "run_archive_checkpoint": RUN_ARCHIVE_CHECKPOINT}
pd.DataFrame([runtime_controls]).T.rename(columns={0: "value"})


## 5. Workspace layout and runtime artifact paths


In [ ]:
REPO_ROOT = Path(os.environ.get("NOTEBOOK14_REPO_ROOT", ".")).resolve()
ARTIFACT_ROOT = Path(os.environ.get("NOTEBOOK14_ARTIFACT_ROOT", REPO_ROOT / "artifacts")).resolve()
NOTEBOOK14_RUNTIME_ROOT = Path(os.environ.get("NOTEBOOK14_RUNTIME_ROOT", ARTIFACT_ROOT / "_notebook_14_runtime")).resolve()
CAMPAIGN_ARTIFACT_ROOT = Path(os.environ.get("NOTEBOOK14_CAMPAIGN_ARTIFACT_ROOT", ARTIFACT_ROOT)).resolve()
EVIDENCE_REVIEW_OUTPUT_DIR = Path(os.environ.get("NOTEBOOK14_EVIDENCE_REVIEW_OUTPUT_DIR", ARTIFACT_ROOT / "_derived" / "evidence_review")).resolve()
GOVERNANCE_OUTPUT_DIR = Path(os.environ.get("NOTEBOOK14_GOVERNANCE_OUTPUT_DIR", ARTIFACT_ROOT / "promotion_governance")).resolve()
CATALOG_LINEAGE_OUTPUT_DIR = Path(os.environ.get("NOTEBOOK14_CATALOG_LINEAGE_OUTPUT_DIR", NOTEBOOK14_RUNTIME_ROOT / "catalog_lineage")).resolve()
SUMMARY_OUTPUT_DIR = Path(os.environ.get("NOTEBOOK14_SUMMARY_OUTPUT_DIR", NOTEBOOK14_RUNTIME_ROOT / "summary")).resolve()
RESTORE_ARCHIVE_ID = os.environ.get("NOTEBOOK14_RESTORE_ARCHIVE_ID", "").strip()
DRIVE_ARCHIVE_ROOT = os.environ.get("NOTEBOOK14_DRIVE_ARCHIVE_ROOT", "").strip()
SELECTED_RUN_ID = os.environ.get("NOTEBOOK14_SELECTED_RUN_ID", "").strip()
SELECTED_CATALOG_ID = os.environ.get("NOTEBOOK14_SELECTED_CATALOG_ID", "").strip()
REVIEW_ID = os.environ.get("NOTEBOOK14_REVIEW_ID", "").strip()
for path in [NOTEBOOK14_RUNTIME_ROOT, SUMMARY_OUTPUT_DIR]:
    ensure_dir(path)
path_summary = {"repo_root": REPO_ROOT, "artifact_root": ARTIFACT_ROOT, "campaign_artifact_root": CAMPAIGN_ARTIFACT_ROOT, "runtime_root": NOTEBOOK14_RUNTIME_ROOT, "evidence_review_output_dir": EVIDENCE_REVIEW_OUTPUT_DIR, "governance_output_dir": GOVERNANCE_OUTPUT_DIR, "catalog_lineage_output_dir": CATALOG_LINEAGE_OUTPUT_DIR, "summary_output_dir": SUMMARY_OUTPUT_DIR, "selected_run_id": SELECTED_RUN_ID or None, "selected_catalog_id": SELECTED_CATALOG_ID or None, "review_id": REVIEW_ID or None}
pd.DataFrame([path_summary]).T.rename(columns={0: "value"})


## 6. Native command discovery

Missing commands are caveats; the notebook does not create replacement logic for unavailable native surfaces.


In [ ]:
CAVEATS: list[str] = list(INSTALL_CAVEATS)
COMMAND_RESULTS: list[dict[str, Any]] = []

def run_command(command: list[str], *, cwd: Path | None = None, timeout: int = 600, allow_run: bool = False, label: str | None = None) -> dict[str, Any]:
    command_label = label or " ".join(command)
    result: dict[str, Any] = {"label": command_label, "command": command, "cwd": cwd.as_posix() if cwd else None, "started_at": utc_now_iso(), "allow_run": allow_run, "returncode": None, "stdout_tail": "", "stderr_tail": "", "duration_seconds": None, "skipped": not allow_run}
    if not allow_run:
        result["completed_at"] = utc_now_iso(); COMMAND_RESULTS.append(result); return result
    started = time.time()
    try:
        completed = subprocess.run(command, cwd=str(cwd) if cwd else None, check=False, capture_output=True, text=True, timeout=timeout)
        result.update({"returncode": completed.returncode, "stdout_tail": tail_text(completed.stdout), "stderr_tail": tail_text(completed.stderr), "duration_seconds": round(time.time() - started, 3), "skipped": False, "completed_at": utc_now_iso()})
    except Exception as exc:
        result.update({"returncode": -1, "stderr_tail": f"{type(exc).__name__}: {exc}", "duration_seconds": round(time.time() - started, 3), "skipped": False, "completed_at": utc_now_iso()})
    COMMAND_RESULTS.append(result); return result

def command_results_dataframe() -> pd.DataFrame:
    return pd.DataFrame([{"label": r.get("label"), "command": " ".join(r.get("command", [])), "returncode": r.get("returncode"), "skipped": r.get("skipped"), "duration_seconds": r.get("duration_seconds"), "stdout_tail_present": bool(r.get("stdout_tail")), "stderr_tail_present": bool(r.get("stderr_tail"))} for r in COMMAND_RESULTS])

NATIVE_COMMANDS = ["stratlake-init-notebook", "stratlake-session-archive-restore-bootstrap", "stratlake-build-evidence-review", "stratlake-run-promotion-governance-report", "stratlake-catalog-index", "stratlake-query-catalog", "stratlake-explore-catalog-evidence", "stratlake-export-catalog-lineage", "stratlake-session-archive-bootstrap"]
command_inventory = []
if PROFILE["DISCOVER_NATIVE_COMMANDS"]:
    for command in NATIVE_COMMANDS:
        available = command_available(command)
        command_inventory.append({"command": command, "available": available, "path": shutil.which(command)})
        if available:
            result = run_command([command, "--help"], cwd=REPO_ROOT, timeout=120, allow_run=True, label=f"{command} --help")
            if result.get("returncode") != 0:
                CAVEATS.append(f"Native command help returned non-zero status: {command}")
        else:
            CAVEATS.append(f"Native command unavailable: {command}")
init_result: dict[str, Any] | None = None
if PROFILE["RUN_STRATLAKE_INIT"] and not RUN_STRATLAKE_INIT:
    CAVEATS.append("Profile requests stratlake init, but NOTEBOOK14_ALLOW_STRATLAKE_INIT=true and RUN_STRATLAKE_INIT=true were not both set.")
if RUN_STRATLAKE_INIT:
    if not command_available("stratlake-init-notebook"):
        CAVEATS.append("StratLake init requested but stratlake-init-notebook is unavailable.")
    else:
        init_result = run_command(["stratlake-init-notebook"], cwd=REPO_ROOT, timeout=600, allow_run=True, label="stratlake init notebook preflight")
        if init_result.get("returncode") != 0:
            CAVEATS.append("StratLake init notebook preflight returned non-zero status.")
command_inventory_df = pd.DataFrame(command_inventory)
command_results_df = command_results_dataframe()
display_markdown("### Native command inventory"); display_df(command_inventory_df, max_rows=20)
display_markdown("### Native command result summary"); display_df(command_results_df, max_rows=30)


## 7. Optional archive restore


In [ ]:
restore_result: dict[str, Any] | None = None
if RUN_ARCHIVE_RESTORE:
    if not RESTORE_ARCHIVE_ID:
        CAVEATS.append("Archive restore requested but NOTEBOOK14_RESTORE_ARCHIVE_ID is empty.")
    elif not command_available("stratlake-session-archive-restore-bootstrap"):
        CAVEATS.append("Archive restore requested but stratlake-session-archive-restore-bootstrap is unavailable.")
    else:
        restore_cmd = ["stratlake-session-archive-restore-bootstrap", "--archive-id", RESTORE_ARCHIVE_ID]
        if DRIVE_ARCHIVE_ROOT:
            restore_cmd.extend(["--drive-archive-root", DRIVE_ARCHIVE_ROOT])
        restore_result = run_command(restore_cmd, cwd=REPO_ROOT, timeout=900, allow_run=True, label="restore Notebook 13 session archive")
        if restore_result.get("returncode") != 0:
            CAVEATS.append("Archive restore command returned non-zero status.")
else:
    CAVEATS.append("Archive restore not run; Notebook 14 is using local/runtime artifact discovery only.")
restore_result


## 8. Discover candidate campaign artifacts

Notebook runtime outputs are inventoried separately and are explicitly excluded from campaign/evidence/governance artifact discovery.


In [ ]:
ARTIFACT_PATTERNS = {
    "campaign_manifest": ["**/campaign*/**/manifest.json", "**/*campaign*manifest*.json"],
    "campaign_summary": ["**/*campaign*summary*.json", "**/*campaign*summary*.csv", "**/*campaign*report*.md"],
    "run_registry": ["**/*run*registry*.json", "**/*registry*.csv", "**/registry*.json"],
    "metrics": ["**/*metrics*.json", "**/*metrics*.csv"],
    "split_metrics": ["**/*split*metrics*.json", "**/*split*metrics*.csv"],
    "promotion_gate": ["**/*promotion*gate*.json", "**/*promotion*gate*.csv"],
    "governance": ["**/*governance*.json", "**/*governance*.csv", "**/*governance*.md"],
    "evidence_review": ["**/_derived/evidence_review/**/*.json", "**/_derived/evidence_review/**/*.csv", "**/_derived/evidence_review/**/*.md"],
}
EXCLUDED_ARTIFACT_PATH_PARTS = {"_notebook_14_runtime", "_notebook_runtime"}
EXCLUDED_ARTIFACT_NAME_PREFIXES = ("notebook_14_", "notebook14_")

def is_notebook_runtime_artifact_path(path: Path) -> bool:
    return bool(set(path.parts) & EXCLUDED_ARTIFACT_PATH_PARTS) or path.name.startswith(EXCLUDED_ARTIFACT_NAME_PREFIXES)

def is_reviewable_artifact_path(path: Path) -> bool:
    return path.is_file() and not is_notebook_runtime_artifact_path(path)

def discover_files(root: Path, patterns: list[str], limit: int = 200) -> list[Path]:
    if not root.exists():
        return []
    found: list[Path] = []
    for pattern in patterns:
        found.extend(path for path in root.glob(pattern) if is_reviewable_artifact_path(path))
    return sorted(set(found), key=lambda p: p.as_posix())[:limit]

def discover_notebook_runtime_outputs(root: Path, limit: int = 200) -> list[Path]:
    if not root.exists():
        return []
    candidates = [p for p in root.rglob("*") if p.is_file() and is_notebook_runtime_artifact_path(p)]
    return sorted(set(candidates), key=lambda p: p.as_posix())[:limit]

artifact_rows: list[dict[str, Any]] = []
for artifact_type, patterns in ARTIFACT_PATTERNS.items():
    for path in discover_files(CAMPAIGN_ARTIFACT_ROOT, patterns):
        stat = path.stat()
        artifact_rows.append({"artifact_type": artifact_type, "path": path.as_posix(), "name": path.name, "suffix": path.suffix, "size_bytes": stat.st_size, "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(), "classification": "reviewable_native_or_upstream_artifact_candidate"})

runtime_artifact_rows: list[dict[str, Any]] = []
for path in discover_notebook_runtime_outputs(CAMPAIGN_ARTIFACT_ROOT):
    stat = path.stat()
    runtime_artifact_rows.append({"artifact_type": "notebook_14_runtime_output", "path": path.as_posix(), "name": path.name, "suffix": path.suffix, "size_bytes": stat.st_size, "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(), "classification": "notebook_runtime_non_canonical_excluded_from_reviewable_artifacts"})

artifact_inventory_df = pd.DataFrame(artifact_rows)
notebook_runtime_inventory_df = pd.DataFrame(runtime_artifact_rows)
if artifact_inventory_df.empty:
    CAVEATS.append(f"No campaign/evidence/governance artifacts discovered under {CAMPAIGN_ARTIFACT_ROOT}. Notebook runtime outputs were excluded from reviewable artifact discovery.")
else:
    display_markdown("### Reviewable artifact candidates"); display_df(artifact_inventory_df, max_rows=50)
if not notebook_runtime_inventory_df.empty:
    display_markdown("### Notebook runtime outputs excluded from reviewable artifact discovery"); display_df(notebook_runtime_inventory_df, max_rows=20)
artifact_inventory_df


## 9. Optional native evidence review pack build and validation


In [ ]:
evidence_build_result: dict[str, Any] | None = None
evidence_validate_result: dict[str, Any] | None = None
def build_evidence_review_command() -> list[str]:
    command = ["stratlake-build-evidence-review", "build", "--artifacts-root", ARTIFACT_ROOT.as_posix()]
    if SELECTED_RUN_ID:
        command.extend(["--selected-run-id", SELECTED_RUN_ID])
    if SELECTED_CATALOG_ID:
        command.extend(["--selected-catalog-id", SELECTED_CATALOG_ID])
    if REVIEW_ID:
        command.extend(["--review-id", REVIEW_ID])
    return command
if RUN_EVIDENCE_REVIEW_PACK_BUILD:
    if not command_available("stratlake-build-evidence-review"):
        CAVEATS.append("Evidence review build requested but stratlake-build-evidence-review is unavailable.")
    elif not SELECTED_RUN_ID and not SELECTED_CATALOG_ID:
        CAVEATS.append("Evidence review build requested but neither NOTEBOOK14_SELECTED_RUN_ID nor NOTEBOOK14_SELECTED_CATALOG_ID is set.")
    else:
        evidence_build_result = run_command(build_evidence_review_command(), cwd=REPO_ROOT, timeout=900, allow_run=True, label="build derived evidence review pack")
        if evidence_build_result.get("returncode") != 0:
            CAVEATS.append("Evidence review pack build returned non-zero status.")
else:
    CAVEATS.append("Native evidence review pack build not run; derived review pack evidence is absent unless previously generated.")
if RUN_EVIDENCE_REVIEW_PACK_VALIDATE:
    validate_cmd = ["stratlake-build-evidence-review", "validate", "--artifacts-root", ARTIFACT_ROOT.as_posix()]
    if REVIEW_ID:
        validate_cmd.extend(["--review-id", REVIEW_ID])
    evidence_validate_result = run_command(validate_cmd, cwd=REPO_ROOT, timeout=600, allow_run=command_available("stratlake-build-evidence-review"), label="validate derived evidence review pack")
    if evidence_validate_result.get("returncode") != 0:
        CAVEATS.append("Evidence review pack validation returned non-zero status.")
else:
    CAVEATS.append("Native evidence review pack validation not run.")
{"build": evidence_build_result, "validate": evidence_validate_result}


## 10. Optional native promotion governance report


In [ ]:
governance_result: dict[str, Any] | None = None
if RUN_PROMOTION_GOVERNANCE_REPORT:
    if not command_available("stratlake-run-promotion-governance-report"):
        CAVEATS.append("Governance report requested but stratlake-run-promotion-governance-report is unavailable.")
    else:
        governance_cmd = ["stratlake-run-promotion-governance-report", "--artifact-root", ARTIFACT_ROOT.as_posix(), "--output-dir", GOVERNANCE_OUTPUT_DIR.as_posix()]
        governance_result = run_command(governance_cmd, cwd=REPO_ROOT, timeout=900, allow_run=True, label="run promotion governance report")
        if governance_result.get("returncode") != 0:
            CAVEATS.append("Promotion governance report returned non-zero status.")
else:
    CAVEATS.append("Native promotion governance report not run; governance readiness is not claimed.")
governance_result


## 11. Optional catalog index/query/evidence/lineage surfaces


In [ ]:
catalog_results: list[dict[str, Any]] = []
if RUN_CATALOG_LINEAGE_EXPORT:
    ensure_dir(CATALOG_LINEAGE_OUTPUT_DIR)
    catalog_commands = [("catalog index", ["stratlake-catalog-index", "--artifact-root", ARTIFACT_ROOT.as_posix()]), ("query catalog", ["stratlake-query-catalog", "--artifact-root", ARTIFACT_ROOT.as_posix()]), ("explore catalog evidence", ["stratlake-explore-catalog-evidence", "--artifact-root", ARTIFACT_ROOT.as_posix()]), ("export catalog lineage", ["stratlake-export-catalog-lineage", "--artifact-root", ARTIFACT_ROOT.as_posix(), "--output-dir", CATALOG_LINEAGE_OUTPUT_DIR.as_posix()])]
    for label, command in catalog_commands:
        if not command_available(command[0]):
            CAVEATS.append(f"Catalog/lineage command unavailable: {command[0]}"); continue
        catalog_results.append(run_command(command, cwd=REPO_ROOT, timeout=900, allow_run=True, label=label))
else:
    CAVEATS.append("Catalog/lineage export not run.")
pd.DataFrame(catalog_results)


## 12. Review-pack and governance artifact inventory


In [ ]:
EXPECTED_EVIDENCE_REVIEW_FILES = ["manifest.json", "review_request.json", "review_summary.json", "catalog_health_diagnostics.json", "validation.json", "selected_record.json", "related_records.json", "resolver_resolution.json", "evidence_index.json", "artifact_inventory.csv", "report.md"]
EXPECTED_GOVERNANCE_FILES = ["promotion_governance_summary.json", "promotion_outcome_matrix.csv", "reason_code_summary.csv", "severity_summary.csv", "workflow_summary.csv", "consistency_validation.json", "promotion_governance_report.md", "manifest.json"]
def latest_matching_dir(root: Path, required_names: list[str]) -> Path | None:
    if not root.exists():
        return None
    scored: list[tuple[int, float, Path]] = []
    for candidate in [p for p in root.rglob("*") if p.is_dir() and not is_notebook_runtime_artifact_path(p)]:
        score = sum((candidate / name).exists() for name in required_names)
        if score:
            scored.append((score, candidate.stat().st_mtime, candidate))
    return sorted(scored, key=lambda item: (item[0], item[1]), reverse=True)[0][2] if scored else None
latest_review_pack_dir = latest_matching_dir(EVIDENCE_REVIEW_OUTPUT_DIR, EXPECTED_EVIDENCE_REVIEW_FILES)
latest_governance_dir = latest_matching_dir(GOVERNANCE_OUTPUT_DIR, EXPECTED_GOVERNANCE_FILES)
review_pack_df = pd.DataFrame([{"expected_file": name, "found": bool(latest_review_pack_dir and (latest_review_pack_dir / name).exists()), "path": (latest_review_pack_dir / name).as_posix() if latest_review_pack_dir and (latest_review_pack_dir / name).exists() else None, "classification": "derived_non_authoritative_write_back_forbidden"} for name in EXPECTED_EVIDENCE_REVIEW_FILES])
governance_files_df = pd.DataFrame([{"expected_file": name, "found": bool(latest_governance_dir and (latest_governance_dir / name).exists()), "path": (latest_governance_dir / name).as_posix() if latest_governance_dir and (latest_governance_dir / name).exists() else None, "classification": "read_only_governance_observability"} for name in EXPECTED_GOVERNANCE_FILES])
if latest_review_pack_dir is None:
    CAVEATS.append("No derived evidence review pack directory discovered.")
if latest_governance_dir is None:
    CAVEATS.append("No promotion governance report directory discovered.")
display_markdown("### Evidence review pack files"); display_df(review_pack_df, max_rows=20)
display_markdown("### Governance report files"); display_df(governance_files_df, max_rows=20)


## 13. Caveat and non-claim register


In [ ]:
NON_CLAIMS = ["production_readiness", "strategy_approval", "promotion_readiness", "governance_readiness", "statistical_significance", "alpha_validation", "split_metric_completeness", "artifact_completeness", "source_runtime_equivalence", "derived_review_pack_is_canonical", "artifact_presence_proves_current_session_execution"]
if artifact_inventory_df.empty:
    CAVEATS.append("Artifact completeness cannot be assessed because no reviewable campaign artifact inventory was discovered. Notebook runtime outputs were excluded.")
split_metric_rows = artifact_inventory_df[artifact_inventory_df["artifact_type"].eq("split_metrics")] if not artifact_inventory_df.empty else pd.DataFrame()
governance_artifact_rows = artifact_inventory_df[artifact_inventory_df["artifact_type"].eq("governance")] if not artifact_inventory_df.empty else pd.DataFrame()
if split_metric_rows.empty:
    CAVEATS.append("No split-metric artifacts discovered; split-metric completeness is not claimed.")
if governance_artifact_rows.empty and latest_governance_dir is None:
    CAVEATS.append("No reviewable governance artifacts discovered; governance readiness is not claimed. Notebook runtime summaries are excluded from governance artifact classification.")
seen = set(); CAVEATS = [c for c in CAVEATS if not (c in seen or seen.add(c))]
audit_register = {"generated_at": utc_now_iso(), "profile": NOTEBOOK14_TEST_PROFILE, "caveat_count": len(CAVEATS), "caveats": CAVEATS, "non_claims": NON_CLAIMS, "derived_review_pack_boundary": "derived_disposable_rebuildable_non_authoritative_write_back_forbidden", "governance_boundary": "read_only_observability_not_promotion_decision", "notebook_runtime_outputs_excluded": True}
audit_register


## 14. Runtime handoff summary


In [ ]:
handoff_summary = {"notebook": "Notebook 14 — StratLake Campaign Evidence Review Pack and Governance Audit", "stance": "notebook_14_staged_cleaned_source_safe_evidence_governance_audit", "generated_at": utc_now_iso(), "profile": NOTEBOOK14_TEST_PROFILE, "runtime_controls": runtime_controls, "paths": path_summary, "command_inventory": command_inventory_df.to_dict(orient="records") if not command_inventory_df.empty else [], "command_results": COMMAND_RESULTS, "artifact_inventory_rows": len(artifact_inventory_df), "notebook_runtime_output_rows": len(notebook_runtime_inventory_df), "notebook_runtime_outputs_excluded_from_reviewable_artifacts": True, "latest_review_pack_dir": latest_review_pack_dir.as_posix() if latest_review_pack_dir else None, "review_pack_file_status": review_pack_df.to_dict(orient="records"), "latest_governance_dir": latest_governance_dir.as_posix() if latest_governance_dir else None, "governance_file_status": governance_files_df.to_dict(orient="records"), "caveats": CAVEATS, "non_claims": NON_CLAIMS}
summary_path = SUMMARY_OUTPUT_DIR / "notebook_14_evidence_governance_handoff_summary.json"
commands_path = SUMMARY_OUTPUT_DIR / "notebook_14_command_results.json"
inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_artifact_inventory.csv"
runtime_inventory_path = SUMMARY_OUTPUT_DIR / "notebook_14_runtime_output_inventory.csv"
caveats_path = SUMMARY_OUTPUT_DIR / "notebook_14_caveats.json"
if PROFILE["WRITE_RUNTIME_SUMMARY"]:
    write_json(summary_path, handoff_summary)
    write_json(commands_path, COMMAND_RESULTS)
    write_json(caveats_path, audit_register)
    if not artifact_inventory_df.empty:
        write_dataframe_csv(inventory_path, artifact_inventory_df)
    if not notebook_runtime_inventory_df.empty:
        write_dataframe_csv(runtime_inventory_path, notebook_runtime_inventory_df)
{"summary_path": summary_path.as_posix(), "commands_path": commands_path.as_posix(), "inventory_path": inventory_path.as_posix() if not artifact_inventory_df.empty else None, "runtime_inventory_path": runtime_inventory_path.as_posix() if not notebook_runtime_inventory_df.empty else None, "caveats_path": caveats_path.as_posix()}


## 15. Final conservative handoff

Notebook 14 is complete for source-safe staging when the notebook is committed with no outputs, execution counts are null, the default profile remains `evidence_governance_preview`, restore/evidence/governance/catalog/checkpoint gates are disabled by default, notebook runtime outputs are excluded from campaign/evidence/governance artifact discovery, no generated review packs/governance artifacts/campaign artifacts/archive restores/logs/screenshots/credentials/private paths/executed outputs are committed, and final claims remain conservative.

Expected M17.1 stance: `notebook_14_staged_cleaned_source_safe_evidence_governance_audit`

Recommended source validation after staging:

```bash
python scripts/check_notebooks_no_outputs.py notebooks
python scripts/validate_repo_cleanliness.py .
python scripts/scan_for_secret_patterns.py .
```
